In [ ]:
# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
#!/usr/bin/env python3
"""
merge_human_and_class.py  —  Google Colab
=========================================
Merge the object-of-study classification (with_tickclass.parquet) with the
HUMAN-VERIFIED inclusion spreadsheet, and emit Excel/CSV for manual review.

WHY: the classification run compared two noisy sources (regex rules vs a mostly
single-model LLM label). Human verification is the arbiter. Merging them lets you
(a) restrict everything to the verified corpus, (b) see where rules and LLM
disagree WITHIN that corpus, and (c) hand-adjudicate a targeted sample instead of
re-reading everything.

OUTPUTS
  1. merged_human_class.parquet   full merge (analysis file)
  2. merged_human_class.xlsx      same, Excel — for eyeballing/filtering
  3. review_disagreements.xlsx    ONLY the rows worth human time, prioritised,
                                  with an empty `human_class` column to fill
  4. printed summary: class distribution INSIDE the verified set, rule-vs-LLM
     agreement inside it, and the top disagreement patterns

SETUP
  !pip install pandas pyarrow openpyxl -q
"""

import re, sys
from pathlib import Path
import pandas as pd

# Regex to remove illegal XML 1.0 characters (which Excel uses)
# This pattern covers control characters from U+0000 to U+0008, U+000B, U+000C,
# U+000E to U+001F, and U+007F to U+009F.
ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
CLASS_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass/with_tickclass.parquet"
HUMAN_XLSX    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen_human_validation.xlsx"
OUT_DIR       = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass"
OUT_PARQUET   = OUT_DIR + "/merged_human_class.parquet"
OUT_XLSX      = OUT_DIR + "/merged_human_class.xlsx"
OUT_CSV      = OUT_DIR + "/merged_human_class.csv"
REVIEW_XLSX   = OUT_DIR + "/review_disagreements.xlsx"

REVIEW_N      = 400        # rows to put in the adjudication sheet
EXCEL_MAX     = 200000     # Excel row ceiling guard (~1,048,576 hard limit)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]
def truthy(x):
    v = S(x).strip().lower()
    return v not in ("", "0", "no", "false", "exclude", "n", "nan", "none")

def clean_string_for_excel(text):
    """Removes illegal characters from a string for Excel compatibility."""
    if isinstance(text, str):
        return ILLEGAL_CHARACTERS_RE.sub('', text)
    return text


# =========================================================================== #
# load + merge                                                                 #
# =========================================================================== #
def load_human():
    hv = pd.read_excel(HUMAN_XLSX, dtype=str)
    hv.columns = [c.strip() for c in hv.columns]
    hv["oa_id"] = hv["oa_id"].map(short_id)
    hv["human_include"] = hv["final_include"].map(truthy)
    keep = [c for c in ["oa_id", "human", "final_include", "human_include", "final_final_include",
                        "match_vs_rule", "llm_label", "llm_agree", "llm_n_relevant",
                        "gpt_label", "gpt_conf", "deepseek_label", "deepseek_conf",
                        "oa_relevance_tier", "oa_relevance_reason"] if c in hv.columns]
    hv = hv[keep].drop_duplicates(subset="oa_id")
    # hv = hv.drop_duplicates(subset="oa_id")
    print(f"[human] {len(hv):,} reviewed rows | included {int(hv['human_include'].sum()):,}",
          file=sys.stderr)
    return hv

def load_class():
    cl = pd.read_parquet(CLASS_PARQUET)
    cl["oa_id"] = cl["oa_id"].map(short_id)
    print(f"[class] {len(cl):,} classified rows", file=sys.stderr)
    return cl


def merge(cl, hv):
    m = cl.merge(hv, on="oa_id", how="left", suffixes=("", "_hv"))
    m["human_include"] = m["human_include"].fillna(False)
    matched = int(m["final_include"].notna().sum())
    print(f"[merge] {len(m):,} rows | matched to a human review: {matched:,} "
          f"({matched/max(len(m),1):.1%})", file=sys.stderr)
    if matched == 0:
        print("[!] NO overlap — check that oa_id formats match on both sides",
              file=sys.stderr)
    # flags that make manual review easy
    m["rule_vs_llm_agree"] = (m["rule_class"].fillna("") == m["tick_object_type"].fillna(""))
    m["needs_review"] = (
        (~m["rule_vs_llm_agree"] & m["rule_class"].fillna("").ne(""))
        | m["tick_object_type"].eq("uncertain")
        | m["class_method"].eq("llm_disagree")
        | m["class_method"].eq("llm_single")     # single-model = unverified
    )
    return m


# =========================================================================== #
# reporting — restricted to the HUMAN-VERIFIED corpus                          #
# =========================================================================== #
def report(m):
    inc = m[m["human_include"]]
    print(f"\n===== INSIDE THE HUMAN-VERIFIED CORPUS (n={len(inc):,}) =====")
    if not len(inc):
        print("[!] empty — check the merge"); return inc
    print("\nclass distribution:")
    vc = inc["tick_object_type"].value_counts()
    for k, v in vc.items():
        print(f"  {k:<26}{v:>8,}  {v/len(inc):>6.1%}")
    print(f"\ntick_examined = true: {int(inc['tick_examined'].sum()):,} "
          f"({inc['tick_examined'].mean():.1%})")
    print("\nlabel source (class_method):")
    print(inc["class_method"].value_counts().to_string())

    both = inc[inc["rule_class"].fillna("").ne("") & inc["class_method"].str.startswith("llm")]
    if len(both):
        print(f"\nrule vs LLM inside verified corpus: agreement "
              f"{(both['rule_class'] == both['tick_object_type']).mean():.1%} "
              f"(n={len(both):,})")
        dis = both[both["rule_class"] != both["tick_object_type"]]
        print("\ntop disagreement patterns (rule -> LLM):")
        pat = (dis.groupby(["rule_class", "tick_object_type"]).size()
                 .sort_values(ascending=False).head(12))
        for (r, l), n in pat.items():
            print(f"  {r:<26} -> {l:<26}{n:>7,}")

    # human-excluded but classified as a real tick class = screener false positive
    exc = m[~m["human_include"] & m["final_include"].notna()]
    if len(exc):
        print(f"\n[cross-check] human-EXCLUDED rows (n={len(exc):,}); "
              f"LLM called {int(exc['tick_object_type'].eq('not_tick').sum()):,} of them "
              f"'not_tick' — agreement with the human exclusion decision.")
    return inc


# =========================================================================== #
# review sheet — only the rows worth a human's time                            #
# =========================================================================== #
def build_review(inc):
    r = inc[inc["needs_review"]].copy()
    if not len(r):
        print("\n[review] nothing flagged", file=sys.stderr); return
    # prioritise: LLM-vs-rule conflicts first, then uncertain, then single-model
    r["_prio"] = (
        (~r["rule_vs_llm_agree"] & r["rule_class"].fillna("").ne("")).astype(int) * 3
        + r["class_method"].eq("llm_disagree").astype(int) * 2
        + r["tick_object_type"].eq("uncertain").astype(int)
    )
    r = r.sort_values("_prio", ascending=False)
    # stratify by the disagreement pattern so no single pattern dominates
    r["_pat"] = r["rule_class"].fillna("") + " -> " + r["tick_object_type"].fillna("")
    per = max(REVIEW_N // max(r["_pat"].nunique(), 1), 5)
    samp = (r.groupby("_pat", group_keys=False)
              .apply(lambda g: g.head(per))
              .head(REVIEW_N))
    cols = [c for c in ["oa_id", "oa_title", "oa_abstract", "oa_journal", "oa_year",
                        "rule_class", "tick_object_type", "llm_openai", "llm_deepseek",
                        "class_method", "class_confidence", "tick_examined",
                        "field_collection", "study_setting", "_pat"] if c in samp.columns]
    out = samp[cols].copy()
    out["human_class"] = ""          # <- coder fills this (use the codebook)
    out["human_notes"] = ""
    out.to_excel(REVIEW_XLSX, index=False)
    print(f"\n[review] {len(out):,} rows -> {REVIEW_XLSX}", file=sys.stderr)
    print("[review] fill `human_class` using tick_inclusion_criteria.md; ideally "
          "TWO coders independently, then compute Cohen's kappa.", file=sys.stderr)


# =========================================================================== #
if __name__ == "__main__":
    cl = load_class()
    hv = load_human()
    m = merge(cl, hv)
    m.to_parquet(OUT_PARQUET, index=False)
    m.to_csv(OUT_CSV, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    inc = report(m)

    # # Excel copy of the merged file (trimmed columns; abstracts make it huge)
    # slim = [c for c in ["oa_id", "oa_title", "oa_journal", "oa_year", "oa_doi",
    #                     "final_include", "human_include", "human", "llm_label",
    #                     "match_vs_rule", "rule_class", "tick_object_type",
    #                     "llm_openai", "llm_deepseek", "class_method",
    #                     "class_confidence", "tick_examined", "field_collection",
    #                     "study_setting", "tick_taxa_llm", "rule_vs_llm_agree",
    #                     "needs_review", "study_bloc", "study_countries"]
    #         if c in m.columns]
    # xl = m[slim]
    # if len(xl) > EXCEL_MAX:
    #     print(f"[xlsx] {len(xl):,} rows > {EXCEL_MAX:,}; writing the verified subset only",
    #           file=sys.stderr)
    #     xl = xl[xl["human_include"]]
    # xl.to_excel(OUT_XLSX, index=False)

    # # Clean string columns for Excel compatibility before writing
    # for col in m.select_dtypes(include=['object']).columns:
    #     m[col] = m[col].apply(clean_string_for_excel)

    # m.to_excel(OUT_XLSX, index=False)
    # print(f"[write] {OUT_XLSX}  ({len(xl):,} rows, {len(slim)} cols)", file=sys.stderr)

    if len(inc):
        build_review(inc)

In [ ]:
m

In [ ]:
inc
